# ⚡ Building an LLM from Scratch with JAX

JAX provides NumPy-compatible arrays with:
- **JIT compilation** (`jax.jit`) — compile functions to XLA for speed
- **Automatic vectorisation** (`jax.vmap`) — apply functions over batches
- **Automatic differentiation** (`jax.grad`) — gradients through any code
- **Pure functional** style — no mutable state

We build a character-level Transformer using **Flax** (JAX's neural network library).

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
# Install if needed: pip install jax flax optax
import jax
import jax.numpy as jnp              # JAX's numpy-compatible array library
from jax import grad, jit, vmap, random
import flax.linen as nn              # Flax neural network modules
import optax                         # JAX-compatible optimisers
import numpy as np
import matplotlib.pyplot as plt
import time, functools

# JAX uses a different random number system — all randomness requires an explicit key
# This design ensures full reproducibility and enables parallel RNG
key = random.PRNGKey(42)             # master random key

print(f'JAX version: {jax.__version__}')
print(f'Default backend: {jax.default_backend()}')
print(f'Devices: {jax.devices()}')

## 1. JAX Fundamentals — What Makes It Different

In [ ]:
# ── JAX Core Concepts ─────────────────────────────────────────────────────────

# 1. JIT compilation: trace the function once, compile to XLA, run fast
@jit
def matrix_multiply(A, B):
    """JIT-compiled matrix multiply — runs on GPU/TPU automatically."""
    return jnp.dot(A, B)

A = jnp.ones((256, 256))
B = jnp.ones((256, 256))
C = matrix_multiply(A, B)   # first call traces + compiles; subsequent calls are fast
print(f'JIT matmul result shape: {C.shape}')

# 2. Automatic differentiation: grad() differentiates through ANY pure function
def quadratic(x):
    """f(x) = 3x² + 2x + 1  →  f'(x) = 6x + 2"""
    return 3 * x**2 + 2 * x + 1

grad_fn = jit(grad(quadratic))    # compile the gradient function
x_val = 2.0
print(f'f(2)  = {quadratic(x_val):.1f}')        # 3*4 + 2*2 + 1 = 17
print(f"f'(2) = {grad_fn(x_val):.1f}  (expected {6*x_val + 2})")  # 14

# 3. vmap: vectorise any function over a batch dimension — no explicit loops
def dot_product(a, b):
    """Single pair dot product."""
    return jnp.dot(a, b)

# Batched version: apply dot_product over first axis of both inputs
batched_dot = vmap(dot_product, in_axes=(0, 0))

key1, key2 = random.split(key)
A_batch = random.normal(key1, (8, 16))   # 8 vectors of dim 16
B_batch = random.normal(key2, (8, 16))
dots = batched_dot(A_batch, B_batch)     # shape (8,) — 8 dot products in parallel
print(f'Batched dot products shape: {dots.shape}')

# 4. Immutable arrays: JAX arrays cannot be modified in-place
x = jnp.array([1.0, 2.0, 3.0])
# Wrong: x[0] = 99  ← this would raise an error
# Correct: use .at[].set() which returns a new array
x_updated = x.at[0].set(99.0)
print(f'Original: {x}, Updated: {x_updated}')  # x is unchanged

## 2. Dataset & Tokenisation

In [ ]:
# ── Same Shakespeare corpus as nb03 ──────────────────────────────────────────
TEXT = """
To be, or not to be, that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles
And by opposing end them. To die—to sleep,
No more; and by a sleep to say we end
The heartache and the thousand natural shocks
All the world's a stage,
And all the men and women merely players;
They have their exits and their entrances,
And one man in his time plays many parts.
""" * 30

# Character-level vocabulary
chars = sorted(set(TEXT))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [char_to_idx[c] for c in s]
decode = lambda ids: ''.join(idx_to_char[i] for i in ids)

# Convert to JAX array (integers)
data_np = np.array(encode(TEXT), dtype=np.int32)
data = jnp.array(data_np)

split = int(0.9 * len(data))
train_data = data[:split]
val_data   = data[split:]

print(f'Vocab: {vocab_size} chars | Train: {len(train_data):,} | Val: {len(val_data):,}')

# Model hyperparameters
BLOCK_SIZE  = 48
BATCH_SIZE  = 16
D_MODEL     = 128
N_HEADS     = 4
N_LAYERS    = 2
FFN_DIM     = 256
DROPOUT     = 0.1
MAX_STEPS   = 800
LR          = 3e-4

def get_batch(key, data, block_size=BLOCK_SIZE, batch_size=BATCH_SIZE):
    """Sample a random batch. JAX-style: takes and splits a PRNGKey."""
    # Generate random starting positions
    ix = random.randint(key, (batch_size,), 0, len(data) - block_size)
    # Gather sequences using fancy indexing
    x = jnp.stack([data[i     : i + block_size    ] for i in ix])
    y = jnp.stack([data[i + 1 : i + block_size + 1] for i in ix])
    return x, y

key, subkey = random.split(key)
xb, yb = get_batch(subkey, train_data)
print(f'Batch shapes: x={xb.shape}, y={yb.shape}')

## 3. Transformer in Flax

Flax uses a functional API: modules are stateless — weights live in a
separate `params` dict that you carry around explicitly.

In [ ]:
# ── Flax Transformer Modules ──────────────────────────────────────────────────

class CausalSelfAttention(nn.Module):
    """
    Multi-head causal self-attention implemented in Flax.
    Flax modules declare attributes (n_heads, d_model) as class fields.
    Weights are created lazily on first call via nn.Dense (= Linear layer).
    """
    n_heads: int
    d_model: int
    dropout_rate: float = 0.0

    @nn.compact
    def __call__(self, x, training=False):
        B, T, C = x.shape
        head_dim = C // self.n_heads

        # Single dense layer projects to Q, K, V simultaneously
        qkv = nn.Dense(3 * C, use_bias=False)(x)      # (B, T, 3C)
        Q, K, V = jnp.split(qkv, 3, axis=-1)          # each (B, T, C)

        # Reshape to (B, n_heads, T, head_dim)
        def split_heads(t):
            return t.reshape(B, T, self.n_heads, head_dim).transpose(0, 2, 1, 3)

        Q, K, V = split_heads(Q), split_heads(K), split_heads(V)

        # Scaled dot-product attention scores
        scale  = head_dim ** -0.5
        scores = jnp.matmul(Q, K.transpose(0, 1, 3, 2)) * scale   # (B, h, T, T)

        # Causal mask: create lower-triangular mask and set future to -1e9
        mask = jnp.tril(jnp.ones((T, T)))
        scores = jnp.where(mask == 0, -1e9, scores)

        # Softmax attention weights + optional dropout
        attn = jax.nn.softmax(scores, axis=-1)
        if training:
            attn = nn.Dropout(self.dropout_rate)(attn, deterministic=False)

        # Weighted sum of values
        out = jnp.matmul(attn, V)                      # (B, h, T, head_dim)

        # Merge heads back to (B, T, C)
        out = out.transpose(0, 2, 1, 3).reshape(B, T, C)

        return nn.Dense(C, use_bias=False)(out)


class TransformerBlock(nn.Module):
    """One Transformer block: pre-norm attention + pre-norm FFN."""
    n_heads: int
    d_model: int
    ffn_dim: int
    dropout_rate: float = 0.0

    @nn.compact
    def __call__(self, x, training=False):
        # Attention sub-layer with residual
        x = x + CausalSelfAttention(self.n_heads, self.d_model, self.dropout_rate)(
            nn.LayerNorm()(x), training=training)
        # FFN sub-layer with residual
        residual = x
        x = nn.LayerNorm()(x)
        x = nn.Dense(self.ffn_dim)(x)
        x = jax.nn.gelu(x)            # GELU activation
        x = nn.Dense(self.d_model)(x)
        if training:
            x = nn.Dropout(self.dropout_rate)(x, deterministic=False)
        return residual + x


class NanoGPT_Flax(nn.Module):
    """Complete GPT model in Flax."""
    vocab_size: int
    d_model:    int
    n_heads:    int
    n_layers:   int
    ffn_dim:    int
    block_size: int
    dropout_rate: float = 0.0

    @nn.compact
    def __call__(self, idx, training=False):
        B, T = idx.shape

        # Token embedding (vocab_size × d_model table)
        tok_emb = nn.Embed(self.vocab_size, self.d_model)(idx)       # (B, T, d_model)

        # Learnable positional embedding
        positions = jnp.arange(T)
        pos_emb   = nn.Embed(self.block_size, self.d_model)(positions)  # (T, d_model)

        x = tok_emb + pos_emb   # broadcast pos_emb over batch dimension

        if training:
            x = nn.Dropout(self.dropout_rate)(x, deterministic=False)

        # Stack transformer blocks
        for _ in range(self.n_layers):
            x = TransformerBlock(
                self.n_heads, self.d_model, self.ffn_dim, self.dropout_rate
            )(x, training=training)

        x = nn.LayerNorm()(x)                   # final normalisation
        logits = nn.Dense(self.vocab_size)(x)   # project to vocab: (B, T, V)
        return logits


# Instantiate and initialise with a dummy input
model = NanoGPT_Flax(
    vocab_size=vocab_size, d_model=D_MODEL, n_heads=N_HEADS,
    n_layers=N_LAYERS, ffn_dim=FFN_DIM, block_size=BLOCK_SIZE, dropout_rate=DROPOUT)

key, init_key = random.split(key)
dummy_input = jnp.zeros((1, BLOCK_SIZE), dtype=jnp.int32)
params = model.init(init_key, dummy_input)   # initialise all weights

n_params = sum(p.size for p in jax.tree_util.tree_leaves(params))
print(f'NanoGPT-Flax parameters: {n_params:,} ({n_params/1e6:.2f}M)')

## 4. Training with Optax

In [ ]:
# ── Training with Optax ───────────────────────────────────────────────────────

# AdamW optimiser with cosine decay learning rate schedule
schedule = optax.cosine_decay_schedule(init_value=LR, decay_steps=MAX_STEPS)
optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),    # gradient clipping
    optax.adamw(learning_rate=schedule, weight_decay=0.01),
)
opt_state = optimizer.init(params)     # initialise optimiser state

def cross_entropy_loss(logits, targets):
    """Standard cross-entropy loss for language modelling."""
    B, T, V = logits.shape
    # Flatten to (B*T, V) and (B*T,) for optax's softmax_cross_entropy
    logits_flat  = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    # One-hot encode targets for optax's API
    one_hot = jax.nn.one_hot(targets_flat, V)
    return optax.softmax_cross_entropy(logits_flat, one_hot).mean()

@jit
def train_step(params, opt_state, x, y, dropout_key):
    """
    One gradient update step — compiled with @jit for speed.
    JAX requires us to explicitly pass the dropout key for randomness.
    """
    def loss_fn(params):
        # Apply model with dropout enabled
        logits = model.apply(params, x, training=True,
                             rngs={'dropout': dropout_key})
        return cross_entropy_loss(logits, y)

    # Compute loss and gradients in one pass
    loss, grads = jax.value_and_grad(loss_fn)(params)

    # Apply gradients via optimiser
    updates, new_opt_state = optimizer.update(grads, opt_state, params)
    new_params = optax.apply_updates(params, updates)

    return new_params, new_opt_state, loss

# Training loop
train_losses = []
start = time.time()

for step in range(MAX_STEPS + 1):
    # Split key for this step (dropout needs its own key)
    key, batch_key, dropout_key = random.split(key, 3)

    x, y = get_batch(batch_key, train_data)

    params, opt_state, loss = train_step(params, opt_state, x, y, dropout_key)

    if step % 200 == 0:
        elapsed = time.time() - start
        print(f'step {step:4d} | loss {float(loss):.4f} | {elapsed:.1f}s')
        train_losses.append(float(loss))

print('Done.')

plt.figure(figsize=(7, 3))
plt.plot(range(0, MAX_STEPS+1, 200), train_losses, color='#34d399', lw=2)
plt.xlabel('Step'); plt.ylabel('Loss'); plt.title('JAX NanoGPT Training')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 5. Text Generation with JAX

In [ ]:
# ── Autoregressive Text Generation in JAX ────────────────────────────────────

def generate(params, seed_text, max_new_tokens=150, temperature=0.8, top_k=15):
    """
    Greedy / temperature-sampled generation.
    JAX's functional style: no mutation — we carry the context as a new array each step.
    """
    context = jnp.array(encode(seed_text), dtype=jnp.int32)[None, :]   # (1, T)
    gen_key = random.PRNGKey(0)

    for _ in range(max_new_tokens):
        # Crop to block_size
        ctx = context[:, -BLOCK_SIZE:]

        # Forward pass (no dropout at inference)
        logits = model.apply(params, ctx, training=False)   # (1, T, V)
        logits = logits[:, -1, :]                           # last position: (1, V)

        # Apply temperature
        logits = logits / temperature

        # Top-k filtering
        if top_k is not None:
            top_vals = jnp.sort(logits, axis=-1)[:, -top_k:-top_k+1]
            logits = jnp.where(logits < top_vals, -jnp.inf, logits)

        # Sample from the distribution
        gen_key, subkey = random.split(gen_key)
        probs   = jax.nn.softmax(logits, axis=-1)
        next_id = random.categorical(subkey, jnp.log(probs + 1e-9))  # (1,)

        # Append to context
        context = jnp.concatenate([context, next_id[:, None]], axis=1)

    return decode(context[0].tolist())

seed = 'To be, or not'
for temp in [0.6, 1.0]:
    print(f'\n--- Temperature {temp} ---')
    print(generate(params, seed, max_new_tokens=100, temperature=temp))

## 6. JAX vs PyTorch: Key Differences

| Aspect | PyTorch | JAX |
|---|---|---|
| State | Module holds weights | Weights in external dict (`params`) |
| Random | Global torch.manual_seed | Explicit PRNGKey splitting |
| Compilation | `torch.compile()` optional | `@jit` encouraged |
| Grad | `loss.backward()` | `jax.grad(fn)(params)` |
| Vectorise | `DataLoader` batching | `vmap` for arbitrary axes |
| Device | `.to(device)` | Automatic via XLA |
| Mutation | In-place ops allowed | Immutable — `.at[].set()` |